In [1]:
from datasets import load_dataset, Audio
from transformers import AutoFeatureExtractor
import numpy as np

In [2]:
ds = load_dataset("sanchit-gandhi/gtzan")
ds = ds["train"].train_test_split(test_size=0.1, shuffle=True)
id2label_fn = ds["train"].features["genre"].int2str

In [ ]:
import gradio as gr


def generate_audio():
    example = ds["train"].shuffle()[0]
    audio = example["audio"]
    return (
        audio["sampling_rate"],
        audio["array"],
    ), id2label_fn(example["genre"])


with gr.Blocks() as demo:
    with gr.Column():
        for _ in range(4):
            audio, label = generate_audio()
            output = gr.Audio(audio, label=label)

demo.launch(debug=True)

In [3]:
model_id = "MIT/ast-finetuned-audioset-10-10-0.4593"

feature_extractor = AutoFeatureExtractor.from_pretrained(model_id, do_normalize=True, return_attention_mask=True)
feature_extractor

ASTFeatureExtractor {
  "do_normalize": true,
  "feature_extractor_type": "ASTFeatureExtractor",
  "feature_size": 1,
  "max_length": 1024,
  "mean": -4.2677393,
  "num_mel_bins": 128,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": true,
  "sampling_rate": 16000,
  "std": 4.5689974
}

In [4]:
ds = ds.cast_column("audio", Audio(sampling_rate=feature_extractor.sampling_rate))

In [5]:
sample = ds['train'][0]["audio"]
print(f"Mean: {np.mean(sample['array']):.3}, Variance: {np.var(sample['array']):.3}")

Mean: -0.000671, Variance: 0.0766


In [6]:
inputs = feature_extractor(sample["array"], sampling_rate=sample["sampling_rate"])

print(f"inputs keys: {list(inputs.keys())}")

print(
    f"Mean: {np.mean(inputs['input_values']):.3}, Variance: {np.var(inputs['input_values']):.3}"
)

inputs keys: ['input_values']
Mean: 0.426, Variance: 0.107


In [7]:
inputs["input_values"]

[array([[-0.21487094, -0.12859462,  0.2482278 , ...,  0.6185667 ,
          0.48825225,  0.58787364],
        [-0.07245539, -0.3235293 ,  0.05329311, ...,  0.5667483 ,
          0.6067169 ,  0.62559587],
        [ 0.00731547, -0.14064042,  0.23618197, ...,  0.5936054 ,
          0.5902543 ,  0.58504945],
        ...,
        [ 0.30166104,  0.05521757,  0.43203998, ...,  0.42240667,
          0.56232125,  0.65693855],
        [ 0.36002386,  0.05785273,  0.43467516, ...,  0.4927926 ,
          0.5868537 ,  0.7675876 ],
        [ 0.25924635,  0.02793042,  0.4047528 , ...,  0.6322809 ,
          0.5402591 ,  0.61492145]], shape=(1024, 128), dtype=float32)]

In [8]:
def preprocess_audio(examples):
    audio_arrays = [x['array'] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=int(feature_extractor.sampling_rate * 30.0),
        truncation=True,
        return_attention_mask=True,
    )
    return inputs

In [9]:
ds_encoded = ds.map(
    preprocess_audio,
    remove_columns=["audio", "file"],
    batched=True,
    batch_size=100,
)

Map:   0%|          | 0/899 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [10]:
ds_encoded = ds_encoded.rename_column("genre", "label")

In [11]:
id2lable = {
    str(i): id2label_fn(i)
    for i in range(len(ds_encoded["train"].features["label"].names))
}
label2id = { v: k for k, v in id2lable.items()}

In [12]:
len(id2lable)

10

In [52]:
from transformers import AutoModelForAudioClassification

num_labels = len(id2lable)


model = AutoModelForAudioClassification.from_pretrained("./best_model")

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

In [ ]:
ds_encoded["train"][0]

In [54]:
from transformers import TrainingArguments, EarlyStoppingCallback


model_name = "ast"
batch_size = 16
gradient_accumulation_steps = 1
num_train_epochs = 10


training_args = TrainingArguments(
    f"{model_name}-finetuned-gtzan",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_train_epochs,
    warmup_steps=100,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
)

In [55]:
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [56]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=ds_encoded["train"],
    eval_dataset=ds_encoded["test"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.376532,1.278372,0.740000
2,0.512891,0.510786,0.820000
3,0.289460,0.437319,0.840000
4,0.126975,0.368898,0.860000
5,0.073912,0.295192,0.880000
6,0.066016,0.311896,0.890000
7,0.005155,0.270875,0.880000
8,0.002635,0.281538,0.870000
9,0.002403,0.256487,0.920000
10,0.001875,0.262344,0.910000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=570, training_loss=0.3229650329422663, metrics={'train_runtime': 824.1132, 'train_samples_per_second': 10.909, 'train_steps_per_second': 0.692, 'total_flos': 6.094112254328832e+17, 'train_loss': 0.3229650329422663, 'epoch': 10.0})

In [57]:
# View the path to the best checkpoint
print(f"Best checkpoint path: {trainer.state.best_model_checkpoint}")

# View the best metric score achieved
print(f"Best metric (Accuracy): {trainer.state.best_metric}")

Best checkpoint path: None
Best metric (Accuracy): None


In [21]:
trainer.save_model("./best_model")
feature_extractor.save_pretrained("./best_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['./best_model/preprocessor_config.json']

In [ ]:
from huggingface_hub import notebook_login

In [27]:
notebook_login()

In [ ]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/kethankrk/ast-finetuned-finetuned-gtzan/commit/7b3853f0d5a130b26e98cd87827f9b902847a910', commit_message='kethankrk/ast-finetuned-gtzan', commit_description='', oid='7b3853f0d5a130b26e98cd87827f9b902847a910', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kethankrk/ast-finetuned-finetuned-gtzan', endpoint='https://huggingface.co', repo_type='model', repo_id='kethankrk/ast-finetuned-finetuned-gtzan'), pr_revision=None, pr_num=None)

In [31]:
feature_extractor.push_to_hub(repo_id="kethankrk/ast-finetuned-finetuned-gtzan")

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/kethankrk/ast-finetuned-finetuned-gtzan/commit/6cb13fb1403a148a2dd246f6177e23df828d3a6e', commit_message='Upload feature extractor', commit_description='', oid='6cb13fb1403a148a2dd246f6177e23df828d3a6e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kethankrk/ast-finetuned-finetuned-gtzan', endpoint='https://huggingface.co', repo_type='model', repo_id='kethankrk/ast-finetuned-finetuned-gtzan'), pr_revision=None, pr_num=None)

In [32]:
from transformers import pipeline

pipe = pipeline("audio-classification","./best_model")

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

In [51]:
idx = 1
data = ds["train"][idx]["audio"]["array"]
true_label = id2label_fn(ds["train"][idx]["genre"])
predictions = pipe(data)
print(f'Pred:\t\t{predictions[0]["label"]} {(predictions[0]["score"] * 100):.3}%\nActual genre:\t{true_label}')

Pred:		disco 99.8%
Actual genre:	disco


In [36]:
id2label_fn(ds["train"][0]["genre"])

'metal'

In [58]:
kwargs = {
    "dataset_tags": "marsyas/gtzan",
    "dataset": "GTZAN",
    "model_name": f"{model_name}-finetuned-gtzan",
    "finetuned_from": model_id,
    "tasks": "audio-classification",
}

trainer.push_to_hub(**kwargs)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/kethankrk/ast-finetuned-gtzan/commit/726eb2a31f664371b89b73f9d0fb509fae8ae959', commit_message='End of training', commit_description='', oid='726eb2a31f664371b89b73f9d0fb509fae8ae959', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kethankrk/ast-finetuned-gtzan', endpoint='https://huggingface.co', repo_type='model', repo_id='kethankrk/ast-finetuned-gtzan'), pr_revision=None, pr_num=None)